In [1]:
import cv2
import glob
import numpy as np
from itertools import product

In [3]:
import cv2
import glob
import numpy as np
from itertools import product

def create_edge_mosaic(input_path, dataset_path, output_path, piece_size=20):
    # 1. 載入並預處理資料集
    dataset_images = []
    dataset_edges = []

    for file in glob.glob(dataset_path):
        img = cv2.imread(file)
        if img is None:
            continue
        img = cv2.resize(img, (piece_size, piece_size))

        # 邊緣偵測
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blurred, 30, 150)

        dataset_images.append(img)
        dataset_edges.append(edges.flatten())

    print(f"載入 {len(dataset_images)} 張圖片")

    # 2. 讀取主圖
    input_img = cv2.imread(input_path)
    h, w = input_img.shape[:2]
    output = np.zeros((h, w, 3), np.uint8)

    # 3. 對每個方塊進行匹配
    for col, row in product(range(w // piece_size), range(h // piece_size)):
        # 取出方塊
        piece = input_img[row*piece_size:(row+1)*piece_size,
                          col*piece_size:(col+1)*piece_size]

        # 邊緣偵測
        gray = cv2.cvtColor(piece, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        piece_edges = cv2.Canny(blurred, 30, 150).flatten()

        # 找最相似邊緣
        distances = [np.linalg.norm(piece_edges - edge) for edge in dataset_edges]
        best_idx = np.argmin(distances)

        # 替換方塊
        output[row*piece_size:(row+1)*piece_size,
               col*piece_size:(col+1)*piece_size] = dataset_images[best_idx]

    cv2.imwrite(output_path, output)
    print(f"完成！輸出到 {output_path}")

# # local執行
# create_edge_mosaic(
#     "assets/hw2_pic1.jpg",
#     "mosaic-master/mosaic-master/dataset/*",
#     "results/hw4_edge_mosaic_output.jpg",
#     piece_size=20
# )

In [5]:
# Colab執行
create_edge_mosaic(
    "/content/hw2_pic1.jpg",
    "/content/dataset/*",
    "/content/hw4_edge_mosaic_output.jpg",
    piece_size=5
)

載入 347 張圖片
完成！輸出到 /content/hw4_edge_mosaic_output.jpg
